# UV and Lunar Enrichment

**Source script:** `enrich_uv_and_moon.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import calendar
import json
import logging
import math
import os
import re
import time
import urllib.request
from collections import Counter
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd




INPUT_PATH = "outputs/imputed_dataset.csv"
OUTPUT_PATH = "outputs/imputed_dataset_enriched.csv"
REPORT_PATH = "outputs/enrichment_report.json"


DATE_COLUMN = "admission_date"
LAT_COLUMN = "lat"
LON_COLUMN = "lon"
ID_COLUMN = None
TIMEZONE_COLUMN = None
ZIP_COLUMN = None
WEATHER_JOB_ID_COLUMN = "weather_job_id"
COUNTRY_COLUMN = None

ROUND_DECIMALS = 3
CACHE_DIR = "./cache_uv"


GEOCODE_CACHE_PATH = "cache_openmeteo/zip_geocode_cache.json"
MANIFEST_PATH = "cache_openmeteo/manifest.csv"
CATS_TO_JOB_PATH = "cache_openmeteo/cats_to_weather_job.csv"


DOWNLOAD_MAX_RETRIES = 5
DOWNLOAD_RETRY_START_SECONDS = 15



UV_REQUEST_MODE = "full_range"



UV_GAP_FILL_ENABLED = True
UV_GAP_FILL_METHOD = "linear"
UV_GAP_FILL_MAX_DAYS = 7


UV_SOURCE = "temis_msr2"


TEMIS_BASE_URL = "https://www.temis.nl/uvradiation/v2.0/msr2/nc"
TEMIS_PREFIX = "uvief"
TEMIS_COVERAGE = "europe"




CAMS_DATASET = "cams-global-atmospheric-composition-forecasts"
CAMS_VARIABLE = "uv_biologically_effective_dose"
CAMS_VARIABLE_IS_INDEX = None
CAMS_TIME_AGG = "max"

CAMS_TIME_LIST = ["00:00", "06:00", "12:00", "18:00"]
CAMS_REQUEST_TEMPLATE = {




    "type": "forecast",
    "leadtime_hour": "0",
    "format": "netcdf_zip",
}


UV_KEYWORDS = ["uv", "uvi", "uv_index", "ultraviolet"]
MOON_KEYWORDS = ["moon", "lunar", "phase"]


REQUIRED_UV_COLS = [
    "uv_index_day0",
    "uv_index_mean_lag1_2",
    "uv_index_mean_lag1_3",
    "uv_index_mean_lag1_7",
    "uv_index_mean_lag1_13",
    "uv_index_peak_lag1_7",
    "uv_index_peak_lag1_14",
]
REQUIRED_MOON_COLS = [
    "moon_phase_fraction_illuminated",
    "moon_phase_angle_deg",
    "moon_phase_category",
    "is_full_moon",
    "is_new_moon",
]


NATIVE_UV_KEYWORDS = ["erythemal", "dose", "irradiance", "uvbed"]


@dataclass
class UVSettings:
    dataset: str
    variable: str
    variable_is_index: Optional[bool]
    time_agg: str
    time_list: Optional[List[str]]
    request_template: Dict[str, object]






def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    return logging.getLogger("enrich_uv_and_moon")


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", name.lower())


def tokenize(name: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", name.lower())


def detect_placeholder_columns(columns: Iterable[str], keywords: List[str]) -> List[str]:
    found = []
    for col in columns:
        col_l = col.lower()
        if any(k in col_l for k in keywords):
            found.append(col)
    return found


def infer_uv_index_flag(variable_name: str) -> Optional[bool]:
    name = variable_name.lower()
    if "uv_index" in name or "uvindex" in name or name == "uvi" or "uvi" in name:
        return True
    if (
        "eryth" in name
        or "dose" in name
        or "irradiance" in name
        or "uvbed" in name
        or "biologically_effective" in name
    ):
        return False
    return None


def find_best_match(
    required: str,
    candidates: List[str],
    patterns: List[str],
    used: set,
    min_score: int,
    exclude_patterns: Optional[List[str]] = None,
) -> Optional[str]:
    from difflib import SequenceMatcher

    req_norm = normalize_name(required)

    def is_allowed(cand_name: str) -> bool:
        if cand_name in used:
            return False
        cand_norm = normalize_name(cand_name)
        if exclude_patterns and any(re.search(p, cand_norm) for p in exclude_patterns):
            return False
        return True


    for pat in patterns:
        matches = []
        for cand in candidates:
            if not is_allowed(cand):
                continue
            cand_norm = normalize_name(cand)
            if re.search(pat, cand_norm):
                matches.append(cand)
        if matches:

            best = max(
                matches,
                key=lambda c: SequenceMatcher(None, req_norm, normalize_name(c)).ratio(),
            )
            return best


    best = None
    best_score = 0
    for cand in candidates:
        if not is_allowed(cand):
            continue
        cand_norm = normalize_name(cand)
        score = int(100 * SequenceMatcher(None, req_norm, cand_norm).ratio())
        if score > best_score:
            best = cand
            best_score = score

    if best_score >= min_score:
        return best
    return None


def map_required_columns(
    required_cols: List[str],
    all_columns: List[str],
    placeholder_columns: List[str],
    patterns_by_required: Dict[str, List[str]],
    logger: logging.Logger,
    exclude_patterns_by_required: Optional[Dict[str, List[str]]] = None,
) -> Dict[str, str]:
    used = set()
    mapping: Dict[str, str] = {}


    all_norm_map = {normalize_name(c): c for c in all_columns}
    for req in required_cols:
        if req in all_columns:
            mapping[req] = req
            used.add(req)
            continue
        req_norm = normalize_name(req)
        if req_norm in all_norm_map and all_norm_map[req_norm] not in used:
            mapping[req] = all_norm_map[req_norm]
            used.add(all_norm_map[req_norm])


    for req in required_cols:
        if req in mapping:
            continue
        cand = find_best_match(
            req,
            placeholder_columns,
            patterns_by_required.get(req, []),
            used,
            min_score=80,
            exclude_patterns=(
                exclude_patterns_by_required.get(req, []) if exclude_patterns_by_required else None
            ),
        )
        if cand:
            mapping[req] = cand
            used.add(cand)


    for req in required_cols:
        if req in mapping:
            continue
        cand = find_best_match(
            req,
            placeholder_columns,
            patterns_by_required.get(req, []),
            used,
            min_score=70,
            exclude_patterns=(
                exclude_patterns_by_required.get(req, []) if exclude_patterns_by_required else None
            ),
        )
        if cand:
            mapping[req] = cand
            used.add(cand)


    for req in required_cols:
        if req not in mapping:
            logger.info("Placeholder not found for %s; will create new column.", req)
            mapping[req] = req

    return mapping


def parse_date_series(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", infer_datetime_format=True, dayfirst=False)
    if dt.isna().any():
        dt2 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True, dayfirst=True)
        dt = dt.fillna(dt2)
    return dt


def auto_detect_date_column(columns: Iterable[str]) -> Optional[str]:
    candidates = [c for c in columns if "date" in c.lower()]
    if not candidates:
        return None
    priority = ["presentation", "admission", "visit", "encounter", "arrival"]
    for p in priority:
        for c in candidates:
            if p in c.lower():
                return c
    if len(candidates) == 1:
        return candidates[0]
    return None


def normalize_zip(value: object) -> Optional[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    if not s:
        return None

    if re.match(r"^\\d+\\.0$", s):
        s = s[:-2]
    s = re.sub(r"[^0-9]", "", s)
    if not s:
        return None
    if len(s) > 5:
        s = s[:5]
    if len(s) < 5:
        s = s.zfill(5)
    return s


def normalize_country(value: object) -> Optional[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    if not s:
        return None
    s_up = s.upper()
    if len(s_up) == 2 and s_up.isalpha():
        return s_up
    country_map = {
        "GERMANY": "DE",
        "DEUTSCHLAND": "DE",
        "UNITED STATES": "US",
        "UNITED STATES OF AMERICA": "US",
        "USA": "US",
    }
    return country_map.get(s_up, None)


def load_zip_geocode_cache(path: Path) -> Tuple[Dict[Tuple[Optional[str], str], Tuple[float, float]], Optional[str]]:
    if not path.exists():
        return {}, None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    mapping: Dict[Tuple[Optional[str], str], Tuple[float, float]] = {}
    countries = set()
    for key, val in data.items():
        if ":" in key:
            country, zip_code = key.split(":", 1)
            country = country.upper()
            countries.add(country)
        else:
            country, zip_code = None, key
        mapping[(country, zip_code)] = (float(val["lat"]), float(val["lon"]))
    default_country = None
    if len(countries) == 1:
        default_country = next(iter(countries))
    return mapping, default_country


def resolve_zip_column(df: pd.DataFrame, explicit: Optional[str]) -> Optional[str]:
    if explicit and explicit in df.columns:
        return explicit
    candidates = [c for c in df.columns if "zip" in c.lower()]
    if not candidates:
        return None

    for c in candidates:
        if normalize_name(c) in ["zip5", "zipcode", "zip_code"]:
            return c
    return candidates[0]


def resolve_weather_job_id_column(df: pd.DataFrame, explicit: Optional[str]) -> Optional[str]:
    if explicit and explicit in df.columns:
        return explicit
    for c in df.columns:
        if normalize_name(c) == "weatherjobid":
            return c
    return None


def resolve_id_column(df: pd.DataFrame, explicit: Optional[str]) -> Optional[str]:
    if explicit and explicit in df.columns:
        return explicit
    for c in df.columns:
        if normalize_name(c) == "id":
            return c
    return None


def valid_lat_lon(lat: float, lon: float) -> bool:
    if lat is None or lon is None:
        return False
    if np.isnan(lat) or np.isnan(lon):
        return False
    return -90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0


def compute_uv_date_for_row(
    dt: Optional[pd.Timestamp],
    tz_str: Optional[str],
    tz_cache: Dict[str, timezone],
    error_counter: Counter,
) -> Optional[date]:
    if dt is None or pd.isna(dt):
        return None
    try:
        if tz_str:
            if tz_str not in tz_cache:
                from zoneinfo import ZoneInfo

                tz_cache[tz_str] = ZoneInfo(tz_str)
            tz = tz_cache[tz_str]
            if dt.tzinfo is None:
                dt_local = dt.replace(tzinfo=tz)
            else:
                dt_local = dt.tz_convert(tz)
            return dt_local.date()

        if dt.tzinfo is not None:
            return dt.tz_convert(timezone.utc).date()
        return dt.date()
    except Exception as exc:
        error_counter[f"timezone_parse_error: {str(exc)}"] += 1
        return None


def compute_moon_date_utc(dt: Optional[pd.Timestamp]) -> Optional[date]:
    if dt is None or pd.isna(dt):
        return None
    if dt.tzinfo is not None:
        return dt.tz_convert(timezone.utc).date()
    return dt.date()


def julian_day(d: date) -> float:
    y = d.year
    m = d.month
    day = d.day
    if m <= 2:
        y -= 1
        m += 12
    a = y // 100
    b = 2 - a + a // 4
    jd = int(365.25 * (y + 4716)) + int(30.6001 * (m + 1)) + day + b - 1524.5
    return jd


def moon_phase(d: date) -> Tuple[float, float, str, int, int]:

    synodic_month = 29.53058867
    new_moon_jd = 2451550.1
    jd = julian_day(d)
    age = (jd - new_moon_jd) % synodic_month
    phase_angle = (360.0 * age / synodic_month) % 360.0
    illumination = 0.5 * (1.0 - math.cos(math.radians(phase_angle)))

    is_new = 1 if illumination < 0.05 else 0
    is_full = 1 if illumination > 0.95 else 0
    if is_new:
        category = "new"
    elif is_full:
        category = "full"
    elif phase_angle < 180.0:
        category = "waxing"
    else:
        category = "waning"

    return illumination, phase_angle, category, is_full, is_new






def sanitize_tag(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def month_range(start: date, end: date) -> List[Tuple[int, int]]:
    months = []
    cur = date(start.year, start.month, 1)
    end_month = date(end.year, end.month, 1)
    while cur <= end_month:
        months.append((cur.year, cur.month))
        if cur.month == 12:
            cur = date(cur.year + 1, 1, 1)
        else:
            cur = date(cur.year, cur.month + 1, 1)
    return months


def build_cams_request(
    settings: UVSettings,
    year: int,
    month: int,
    north: float,
    west: float,
    south: float,
    east: float,
) -> Dict[str, object]:
    last_day = calendar.monthrange(year, month)[1]
    req = dict(settings.request_template)
    req["variable"] = settings.variable
    req["date"] = f"{year}-{month:02d}-01/{year}-{month:02d}-{last_day:02d}"
    req["area"] = [north, west, south, east]
    if settings.time_list is not None:
        req["time"] = settings.time_list
    return req


def build_cams_request_range(
    settings: UVSettings,
    start_date: date,
    end_date: date,
    north: float,
    west: float,
    south: float,
    east: float,
) -> Dict[str, object]:
    req = dict(settings.request_template)
    req["variable"] = settings.variable
    req["date"] = f"{start_date.isoformat()}/{end_date.isoformat()}"
    req["area"] = [north, west, south, east]
    if settings.time_list is not None:
        req["time"] = settings.time_list
    return req


def open_dataset_and_extract_daily(
    nc_path: Path,
    variable: str,
    lat: float,
    lon: float,
    time_agg: str,
    error_counter: Counter,
) -> Optional[pd.Series]:
    try:
        import xarray as xr
    except Exception as exc:
        error_counter[f"xarray_import_error: {str(exc)}"] += 1
        return None

    try:
        ds = xr.open_dataset(nc_path)
    except Exception as exc:
        error_counter[f"netcdf_open_error: {str(exc)}"] += 1
        return None

    try:
        if variable not in ds.data_vars:

            candidates = list(ds.data_vars.keys())
            error_counter[f"cams_variable_not_found: {variable} (available: {candidates})"] += 1
            return None

        var = ds[variable]


        lat_name = None
        lon_name = None
        for cand in ["latitude", "lat", "y"]:
            if cand in var.coords:
                lat_name = cand
                break
        for cand in ["longitude", "lon", "x"]:
            if cand in var.coords:
                lon_name = cand
                break
        if lat_name is None or lon_name is None:
            error_counter["cams_lat_lon_coords_not_found"] += 1
            return None


        for dim in ["step", "expver", "number"]:
            if dim in var.dims:
                var = var.isel({dim: 0})

        try:
            var_point = var.sel({lat_name: lat, lon_name: lon}, method="nearest")
        except Exception as exc:
            error_counter[f"cams_point_select_error: {str(exc)}"] += 1
            return None


        time_dim = None
        for cand in ["time", "valid_time", "date"]:
            if cand in var_point.coords:
                time_dim = cand
                break

        try:
            if time_dim:
                series = var_point.to_series()
                series.index = pd.to_datetime(series.index)
                if series.index.tz is not None:
                    series.index = series.index.tz_convert(None)
                if time_dim in ["time", "valid_time"]:
                    if time_agg == "mean":
                        daily = series.resample("1D").mean()
                    else:
                        daily = series.resample("1D").max()
                else:
                    daily = series
            else:
                error_counter["cams_time_dimension_not_found"] += 1
                return None
        except Exception as exc:
            error_counter[f"cams_time_processing_error: {str(exc)}"] += 1
            return None

        return daily
    finally:
        try:
            ds.close()
        except Exception:
            pass


def load_or_download_month(
    lat: float,
    lon: float,
    year: int,
    month: int,
    settings: UVSettings,
    cache_dir: Path,
    error_counter: Counter,
    logger: logging.Logger,
) -> Optional[pd.Series]:
    cache_dir.mkdir(parents=True, exist_ok=True)

    dataset_tag = sanitize_tag(settings.dataset)
    var_tag = sanitize_tag(settings.variable)
    lat_str = f"{lat:.{ROUND_DECIMALS}f}"
    lon_str = f"{lon:.{ROUND_DECIMALS}f}"
    cache_file = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{year}{month:02d}.csv"

    if cache_file.exists():
        try:
            df = pd.read_csv(cache_file)
            df["date"] = pd.to_datetime(df["date"]).dt.date
            return pd.Series(df["value"].values, index=pd.to_datetime(df["date"]))
        except Exception as exc:
            error_counter[f"cache_read_error: {str(exc)}"] += 1


    try:
        import cdsapi
    except Exception as exc:
        error_counter[f"cdsapi_import_error: {str(exc)}"] += 1
        return None

    buffer_deg = 0.25
    north = min(lat + buffer_deg, 90.0)
    south = max(lat - buffer_deg, -90.0)
    west = max(lon - buffer_deg, -180.0)
    east = min(lon + buffer_deg, 180.0)

    request = build_cams_request(settings, year, month, north, west, south, east)


    fmt = None
    if "format" in settings.request_template:
        fmt = str(settings.request_template.get("format"))
    elif "data_format" in settings.request_template:
        fmt = str(settings.request_template.get("data_format"))
    fmt = (fmt or "netcdf").lower()

    if "zip" in fmt:
        download_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{year}{month:02d}.zip"
        extract_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{year}{month:02d}.nc"
    else:
        download_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{year}{month:02d}.nc"
        extract_path = download_path

    logger.info("Downloading CAMS UV for %s %s, %04d-%02d", lat_str, lon_str, year, month)

    try:
        client = cdsapi.Client()
    except Exception as exc:
        error_counter[f"cds_client_init_error: {str(exc)}"] += 1
        return None

    success = False
    last_exc: Optional[Exception] = None
    for attempt in range(1, DOWNLOAD_MAX_RETRIES + 1):
        try:
            client.retrieve(settings.dataset, request, str(download_path))
            success = True
            break
        except Exception as exc:
            last_exc = exc
            wait_seconds = DOWNLOAD_RETRY_START_SECONDS * (2 ** (attempt - 1))
            logger.warning(
                "CAMS download failed (attempt %d/%d). Retrying in %ds. Error: %s",
                attempt,
                DOWNLOAD_MAX_RETRIES,
                wait_seconds,
                str(exc),
            )
            time.sleep(wait_seconds)

    if not success:
        error_counter[f"cds_download_error: {str(last_exc)}"] += 1
        return None


    if download_path.suffix == ".zip":
        try:
            import zipfile

            with zipfile.ZipFile(download_path, "r") as zf:
                nc_files = [n for n in zf.namelist() if n.endswith(".nc")]
                if not nc_files:
                    error_counter["zip_no_netcdf_found"] += 1
                    return None
                zf.extract(nc_files[0], path=cache_dir)
                extracted = cache_dir / nc_files[0]

                extracted.replace(extract_path)
        except Exception as exc:
            error_counter[f"zip_extract_error: {str(exc)}"] += 1
            return None

    daily = open_dataset_and_extract_daily(
        extract_path, settings.variable, lat, lon, settings.time_agg, error_counter
    )
    if daily is None:
        return None


    try:
        df_out = pd.DataFrame({"date": daily.index.date, "value": daily.values})
        df_out.to_csv(cache_file, index=False)
    except Exception as exc:
        error_counter[f"cache_write_error: {str(exc)}"] += 1

    return daily


def load_or_download_range(
    lat: float,
    lon: float,
    start_date: date,
    end_date: date,
    settings: UVSettings,
    cache_dir: Path,
    error_counter: Counter,
    logger: logging.Logger,
) -> Optional[pd.Series]:
    cache_dir.mkdir(parents=True, exist_ok=True)

    dataset_tag = sanitize_tag(settings.dataset)
    var_tag = sanitize_tag(settings.variable)
    lat_str = f"{lat:.{ROUND_DECIMALS}f}"
    lon_str = f"{lon:.{ROUND_DECIMALS}f}"
    date_tag = f"{start_date.isoformat()}_{end_date.isoformat()}"
    cache_file = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{date_tag}.csv"

    if cache_file.exists():
        try:
            df = pd.read_csv(cache_file)
            df["date"] = pd.to_datetime(df["date"]).dt.date
            return pd.Series(df["value"].values, index=pd.to_datetime(df["date"]))
        except Exception as exc:
            error_counter[f"cache_read_error: {str(exc)}"] += 1

    try:
        import cdsapi
    except Exception as exc:
        error_counter[f"cdsapi_import_error: {str(exc)}"] += 1
        return None

    buffer_deg = 0.25
    north = min(lat + buffer_deg, 90.0)
    south = max(lat - buffer_deg, -90.0)
    west = max(lon - buffer_deg, -180.0)
    east = min(lon + buffer_deg, 180.0)

    request = build_cams_request_range(settings, start_date, end_date, north, west, south, east)

    fmt = None
    if "format" in settings.request_template:
        fmt = str(settings.request_template.get("format"))
    elif "data_format" in settings.request_template:
        fmt = str(settings.request_template.get("data_format"))
    fmt = (fmt or "netcdf").lower()

    if "zip" in fmt:
        download_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{date_tag}.zip"
        extract_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{date_tag}.nc"
    else:
        download_path = cache_dir / f"{dataset_tag}_{var_tag}_{lat_str}_{lon_str}_{date_tag}.nc"
        extract_path = download_path

    logger.info(
        "Downloading CAMS UV for %s %s, %s -> %s",
        lat_str,
        lon_str,
        start_date.isoformat(),
        end_date.isoformat(),
    )

    try:
        client = cdsapi.Client()
    except Exception as exc:
        error_counter[f"cds_client_init_error: {str(exc)}"] += 1
        return None

    success = False
    last_exc: Optional[Exception] = None
    for attempt in range(1, DOWNLOAD_MAX_RETRIES + 1):
        try:
            client.retrieve(settings.dataset, request, str(download_path))
            success = True
            break
        except Exception as exc:
            last_exc = exc
            wait_seconds = DOWNLOAD_RETRY_START_SECONDS * (2 ** (attempt - 1))
            logger.warning(
                "CAMS download failed (attempt %d/%d). Retrying in %ds. Error: %s",
                attempt,
                DOWNLOAD_MAX_RETRIES,
                wait_seconds,
                str(exc),
            )
            time.sleep(wait_seconds)

    if not success:
        error_counter[f"cds_download_error: {str(last_exc)}"] += 1
        return None

    if download_path.suffix == ".zip":
        try:
            import zipfile

            with zipfile.ZipFile(download_path, "r") as zf:
                nc_files = [n for n in zf.namelist() if n.endswith(".nc")]
                if not nc_files:
                    error_counter["zip_no_netcdf_found"] += 1
                    return None
                zf.extract(nc_files[0], path=cache_dir)
                extracted = cache_dir / nc_files[0]
                extracted.replace(extract_path)
        except Exception as exc:
            error_counter[f"zip_extract_error: {str(exc)}"] += 1
            return None

    daily = open_dataset_and_extract_daily(
        extract_path, settings.variable, lat, lon, settings.time_agg, error_counter
    )
    if daily is None:
        return None

    try:
        df_out = pd.DataFrame({"date": daily.index.date, "value": daily.values})
        df_out.to_csv(cache_file, index=False)
    except Exception as exc:
        error_counter[f"cache_write_error: {str(exc)}"] += 1

    return daily


def get_uv_series_cams(
    lat: float,
    lon: float,
    min_date: date,
    max_date: date,
    settings: UVSettings,
    cache_dir: Path,
    error_counter: Counter,
    logger: logging.Logger,
    request_mode: str,
) -> Dict[date, float]:
    series_map: Dict[date, float] = {}
    if request_mode == "full_range":
        daily = load_or_download_range(
            lat, lon, min_date, max_date, settings, cache_dir, error_counter, logger
        )
        if daily is not None:
            for dt_idx, value in daily.items():
                d = pd.to_datetime(dt_idx).date()
                series_map[d] = float(value) if not pd.isna(value) else np.nan
        return series_map

    for year, month in month_range(min_date, max_date):
        daily = load_or_download_month(
            lat, lon, year, month, settings, cache_dir, error_counter, logger
        )
        if daily is None:
            continue
        for dt_idx, value in daily.items():
            d = pd.to_datetime(dt_idx).date()
            series_map[d] = float(value) if not pd.isna(value) else np.nan
    return series_map


def nanmean(values: List[Optional[float]]) -> float:
    vals = [v for v in values if v is not None and not pd.isna(v)]
    if not vals:
        return np.nan
    return float(np.mean(vals))


def nanmax(values: List[Optional[float]]) -> float:
    vals = [v for v in values if v is not None and not pd.isna(v)]
    if not vals:
        return np.nan
    return float(np.max(vals))







def temis_year_url(base_url: str, prefix: str, year: int, coverage: str) -> str:
    return f"{base_url}/{year}/{prefix}{year}_msr_{coverage}.nc"


def download_temis_year_file(
    base_url: str,
    prefix: str,
    year: int,
    coverage: str,
    cache_dir: Path,
    error_counter: Counter,
    logger: logging.Logger,
) -> Optional[Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    filename = f"{prefix}{year}_msr_{coverage}.nc"
    dest = cache_dir / filename
    if dest.exists():
        return dest

    url = temis_year_url(base_url, prefix, year, coverage)
    logger.info("Downloading TEMIS MSR-2 file: %s", url)

    last_exc: Optional[Exception] = None
    for attempt in range(1, DOWNLOAD_MAX_RETRIES + 1):
        try:
            urllib.request.urlretrieve(url, dest)
            return dest
        except Exception as exc:
            last_exc = exc
            wait_seconds = DOWNLOAD_RETRY_START_SECONDS * (2 ** (attempt - 1))
            logger.warning(
                "TEMIS download failed (attempt %d/%d). Retrying in %ds. Error: %s",
                attempt,
                DOWNLOAD_MAX_RETRIES,
                wait_seconds,
                str(exc),
            )
            time.sleep(wait_seconds)

    error_counter[f"temis_download_error: {str(last_exc)}"] += 1
    return None


def find_coord_name_from_ds(ds, candidates: List[str]) -> Optional[str]:
    for cand in candidates:
        if cand in ds.coords:
            return cand
        if cand in ds.variables:
            return cand

    for name, var in ds.variables.items():
        units = str(var.attrs.get("units", "")).lower()
        if "degrees_north" in units or "degree_north" in units:
            return name
        if "degrees_east" in units or "degree_east" in units:
            return name
    return None


def select_temis_variable(ds, prefix: str) -> Optional[str]:
    candidates = [
        prefix,
        prefix.lower(),
        prefix.upper(),
        "uv_index",
        "uvindex",
        "uv_ind",
        "uv",
    ]
    for cand in candidates:
        if cand in ds.data_vars:
            return cand
        if cand.upper() in ds.data_vars:
            return cand.upper()
        if cand.lower() in ds.data_vars:
            return cand.lower()
    for var in ds.data_vars:
        name = var.lower()
        if any(k in name for k in ["lat", "lon", "time", "date"]):
            continue
        return var
    return None


def extract_time_index_from_ds(ds, var_point) -> Optional[pd.DatetimeIndex]:
    if "time" in var_point.coords:
        times = pd.to_datetime(var_point["time"].values)
        return pd.DatetimeIndex(times)
    for key in ["date", "yyyymmdd", "YYYYMMDD"]:
        if key in ds.variables:
            vals = np.asarray(ds[key].values).reshape(-1)
            vals = vals.astype(int)
            dates = [datetime.strptime(str(v), "%Y%m%d") for v in vals]
            return pd.DatetimeIndex(dates)
    return None


def open_temis_dataset_extract_daily(
    nc_path: Path,
    prefix: str,
    lat: float,
    lon: float,
    error_counter: Counter,
) -> Optional[pd.Series]:
    try:
        import xarray as xr
    except Exception as exc:
        error_counter[f"xarray_import_error: {str(exc)}"] += 1
        return None

    group = None
    try:
        from netCDF4 import Dataset

        with Dataset(nc_path) as nc:
            if "PRODUCT" in nc.groups:
                group = "PRODUCT"
    except Exception:
        group = None

    try:
        ds = xr.open_dataset(nc_path, group=group)
    except Exception as exc:
        error_counter[f"netcdf_open_error: {str(exc)}"] += 1
        return None

    try:
        var_name = select_temis_variable(ds, prefix)
        if not var_name:
            error_counter["temis_variable_not_found"] += 1
            return None

        var = ds[var_name]
        lat_name = find_coord_name_from_ds(ds, ["latitude", "lat", "y"])
        lon_name = find_coord_name_from_ds(ds, ["longitude", "lon", "x"])
        if lat_name is None or lon_name is None:
            error_counter["temis_lat_lon_coords_not_found"] += 1
            return None


        for dim in ["level", "lev", "height"]:
            if dim in var.dims:
                var = var.isel({dim: 0})

        try:
            var_point = var.sel({lat_name: lat, lon_name: lon}, method="nearest")
        except Exception as exc:
            error_counter[f"temis_point_select_error: {str(exc)}"] += 1
            return None

        time_index = extract_time_index_from_ds(ds, var_point)
        if time_index is None:
            error_counter["temis_time_dimension_not_found"] += 1
            return None

        values = np.asarray(var_point.values).reshape(-1)
        series = pd.Series(values, index=time_index)
        return series
    finally:
        try:
            ds.close()
        except Exception:
            pass


def get_uv_series_temis(
    lat: float,
    lon: float,
    min_date: date,
    max_date: date,
    base_url: str,
    prefix: str,
    coverage: str,
    cache_dir: Path,
    error_counter: Counter,
    logger: logging.Logger,
    gap_fill_enabled: bool,
    gap_fill_max_days: int,
) -> Tuple[Dict[date, float], int]:
    series_map: Dict[date, float] = {}
    filled_count = 0
    lat_str = f"{lat:.{ROUND_DECIMALS}f}"
    lon_str = f"{lon:.{ROUND_DECIMALS}f}"

    years = range(min_date.year, max_date.year + 1)
    loc_cache_dir = cache_dir / "temis_msr2"
    for year in years:
        loc_csv = loc_cache_dir / f"uv_{lat_str}_{lon_str}_{year}.csv"
        if loc_csv.exists():
            try:
                df = pd.read_csv(loc_csv)
                for d, v in zip(pd.to_datetime(df["date"]).dt.date, df["value"].values):
                    series_map[d] = float(v) if not pd.isna(v) else np.nan
                continue
            except Exception as exc:
                error_counter[f"cache_read_error: {str(exc)}"] += 1

        year_dir = loc_cache_dir / str(year)
        nc_path = download_temis_year_file(
            base_url, prefix, year, coverage, year_dir, error_counter, logger
        )
        if not nc_path:
            continue

        series = open_temis_dataset_extract_daily(
            nc_path, prefix, lat, lon, error_counter
        )
        if series is None:
            continue

        df_out = pd.DataFrame(
            {"date": series.index.date, "value": series.values}
        )
        try:
            loc_cache_dir.mkdir(parents=True, exist_ok=True)
            df_out.to_csv(loc_csv, index=False)
        except Exception as exc:
            error_counter[f"cache_write_error: {str(exc)}"] += 1

        for d, v in zip(series.index.date, series.values):
            series_map[d] = float(v) if not pd.isna(v) else np.nan

    if gap_fill_enabled and series_map:
        idx = pd.date_range(min_date, max_date, freq="D")
        ser = pd.Series(index=idx, dtype="float64")
        for d, v in series_map.items():
            ser.loc[pd.to_datetime(d)] = v
        before_missing = int(ser.isna().sum())
        if UV_GAP_FILL_METHOD == "linear":
            ser = ser.interpolate(method="time", limit=gap_fill_max_days, limit_area="inside")
        after_missing = int(ser.isna().sum())
        filled_count = max(0, before_missing - after_missing)
        series_map = {d.date(): (float(v) if not pd.isna(v) else np.nan) for d, v in ser.items()}

    return series_map, filled_count






def build_uv_patterns() -> Dict[str, List[str]]:
    return {
        "uv_index_day0": [
            r"uv.*(day0|day_0|d0)",
            r"uv.*(day1|day_1|d1)",
        ],
        "uv_index_mean_lag1_2": [
            r"uv.*(mean).*?(lag1_2|lag1-2|1_2|1-2)",
            r"uv.*(mean).*?(mean_2d|mean2d|mean_2|mean2|2d)",
        ],
        "uv_index_mean_lag1_3": [
            r"uv.*(mean).*?(lag1_3|lag1-3|1_3|1-3)",
            r"uv.*(mean).*?(mean_3d|mean3d|mean_3|mean3|3d)",
        ],
        "uv_index_mean_lag1_7": [
            r"uv.*(mean).*?(lag1_7|lag1-7|1_7|1-7)",
            r"uv.*(mean).*?(mean_7d|mean7d|mean_7|mean7|7d)",
        ],
        "uv_index_mean_lag1_13": [
            r"uv.*(mean).*?(lag1_13|lag1-13|1_13|1-13)",
            r"uv.*(mean).*?(mean_13d|mean13d|mean_13|mean13|13d)",
        ],
        "uv_index_peak_lag1_7": [
            r"uv.*(peak|max).*?(lag1_7|lag1-7|1_7|1-7)",
            r"uv.*(peak|max).*?(peak_7d|max_7d|peak7d|max7d|7d)",
        ],
        "uv_index_peak_lag1_14": [
            r"uv.*(peak|max).*?(lag1_14|lag1-14|1_14|1-14)",
            r"uv.*(peak|max).*?(peak_14d|max_14d|peak14d|max14d|14d)",
        ],
    }


def build_moon_patterns() -> Dict[str, List[str]]:
    return {
        "moon_phase_fraction_illuminated": [r"moon.*(fraction|illum)", r"fraction.*moon", r"illum.*moon"],
        "moon_phase_angle_deg": [r"moon.*(angle|deg)", r"angle.*moon"],
        "moon_phase_category": [r"moon.*(category|phase)", r"phase.*moon"],
        "is_full_moon": [r"full.*moon", r"moon.*full", r"isfull"],
        "is_new_moon": [r"new.*moon", r"moon.*new", r"isnew"],
    }


def build_uv_exclusions() -> Dict[str, List[str]]:

    return {
        "uv_index_mean_lag1_2": [r"3d", r"7d", r"13d", r"14d"],
        "uv_index_mean_lag1_3": [r"13d", r"14d"],
        "uv_index_mean_lag1_7": [r"13d", r"14d"],
        "uv_index_mean_lag1_13": [r"14d"],
        "uv_index_peak_lag1_7": [r"14d"],
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Enrich dataset with CAMS UV and moon phase features.")
    parser.add_argument("--input-path", default=INPUT_PATH)
    parser.add_argument("--output-path", default=OUTPUT_PATH)
    parser.add_argument("--report-path", default=REPORT_PATH)
    parser.add_argument("--date-column", default=DATE_COLUMN)
    parser.add_argument("--lat-column", default=LAT_COLUMN)
    parser.add_argument("--lon-column", default=LON_COLUMN)
    parser.add_argument("--id-column", default=ID_COLUMN)
    parser.add_argument("--timezone-column", default=TIMEZONE_COLUMN)
    parser.add_argument("--zip-column", default=ZIP_COLUMN)
    parser.add_argument("--weather-job-id-column", default=WEATHER_JOB_ID_COLUMN)
    parser.add_argument("--country-column", default=COUNTRY_COLUMN)
    parser.add_argument("--geocode-cache-path", default=GEOCODE_CACHE_PATH)
    parser.add_argument("--manifest-path", default=MANIFEST_PATH)
    parser.add_argument("--cats-to-job-path", default=CATS_TO_JOB_PATH)
    parser.add_argument("--cache-dir", default=CACHE_DIR)
    parser.add_argument("--round-decimals", type=int, default=ROUND_DECIMALS)
    parser.add_argument("--uv-source", default=UV_SOURCE, choices=["temis_msr2", "cams"])
    parser.add_argument("--temis-base-url", default=TEMIS_BASE_URL)
    parser.add_argument("--temis-prefix", default=TEMIS_PREFIX)
    parser.add_argument("--temis-coverage", default=TEMIS_COVERAGE, choices=["europe", "world"])
    parser.add_argument("--uv-gap-fill-enabled", action="store_true", default=UV_GAP_FILL_ENABLED)
    parser.add_argument("--uv-gap-fill-disabled", action="store_true", default=False)
    parser.add_argument("--uv-gap-fill-max-days", type=int, default=UV_GAP_FILL_MAX_DAYS)
    parser.add_argument("--cams-dataset", default=CAMS_DATASET)
    parser.add_argument("--cams-variable", default=CAMS_VARIABLE)
    parser.add_argument("--cams-variable-is-index", default=CAMS_VARIABLE_IS_INDEX)
    parser.add_argument("--cams-time-agg", default=CAMS_TIME_AGG, choices=["max", "mean"])
    parser.add_argument(
        "--uv-request-mode",
        default=UV_REQUEST_MODE,
        choices=["full_range", "monthly"],
        help="Request UV data in one big chunk per location or by month.",
    )
    return parser.parse_args()


def main() -> None:
    logger = setup_logging()
    error_counter: Counter = Counter()

    args = parse_args()
    input_path = Path(args.input_path)
    output_path = Path(args.output_path)
    report_path = Path(args.report_path)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")


    input_ext = input_path.suffix.lower()
    if input_ext in [".xlsx", ".xls"]:
        xl = pd.ExcelFile(input_path)
        sheet_name = xl.sheet_names[0]
        df = xl.parse(sheet_name)
        input_is_xlsx = True
    elif input_ext == ".csv":
        df = pd.read_csv(input_path)
        sheet_name = None
        input_is_xlsx = False
    else:
        raise ValueError("Input file must be .csv or .xlsx/.xls")

    logger.info("Loaded dataset with %d rows and %d columns", len(df), len(df.columns))
    input_columns_count = int(len(df.columns))


    uv_placeholders = detect_placeholder_columns(df.columns, UV_KEYWORDS)
    moon_placeholders = detect_placeholder_columns(df.columns, MOON_KEYWORDS)

    logger.info("Detected UV placeholders: %s", uv_placeholders)
    logger.info("Detected moon placeholders: %s", moon_placeholders)


    uv_mapping = map_required_columns(
        REQUIRED_UV_COLS,
        list(df.columns),
        uv_placeholders,
        build_uv_patterns(),
        logger,
        exclude_patterns_by_required=build_uv_exclusions(),
    )
    moon_mapping = map_required_columns(
        REQUIRED_MOON_COLS,
        list(df.columns),
        moon_placeholders,
        build_moon_patterns(),
        logger,
    )

    required_to_actual = {**uv_mapping, **moon_mapping}

    uv_mapping_notes = []
    day0_actual = uv_mapping.get("uv_index_day0")
    if day0_actual:
        day0_norm = normalize_name(day0_actual)
        if "day1" in day0_norm and "day0" not in day0_norm:
            uv_mapping_notes.append(
                f"uv_index_day0 mapped to column '{day0_actual}' containing day1; confirm semantics."
            )
    mean_2_actual = uv_mapping.get("uv_index_mean_lag1_2")
    if mean_2_actual:
        mean_norm = normalize_name(mean_2_actual)
        if any(x in mean_norm for x in ["3d", "7d", "13d", "14d"]) and "2d" not in mean_norm:
            uv_mapping_notes.append(
                f"uv_index_mean_lag1_2 mapped to column '{mean_2_actual}' with non-2d label; confirm semantics."
            )

    mean_3_actual = uv_mapping.get("uv_index_mean_lag1_3")
    if mean_3_actual:
        mean_norm = normalize_name(mean_3_actual)
        if "2d" in mean_norm and "3d" not in mean_norm:
            uv_mapping_notes.append(
                f"uv_index_mean_lag1_3 mapped to column '{mean_3_actual}' containing 2d; confirm semantics."
            )

    mean_13_actual = uv_mapping.get("uv_index_mean_lag1_13")
    if mean_13_actual:
        mean_norm = normalize_name(mean_13_actual)
        if "14d" in mean_norm and "13d" not in mean_norm:
            uv_mapping_notes.append(
                f"uv_index_mean_lag1_13 mapped to column '{mean_13_actual}' containing 14d; confirm semantics."
            )


    for req, actual in required_to_actual.items():
        if actual not in df.columns:
            df[actual] = np.nan


    for req, actual in required_to_actual.items():
        if actual in df.columns and df[actual].notna().any():
            logger.warning("Column %s has existing values; they will be overwritten.", actual)


    date_col_used = args.date_column
    if date_col_used not in df.columns:
        detected = auto_detect_date_column(df.columns)
        if detected:
            logger.warning(
                "Date column %s not found; using detected column: %s",
                args.date_column,
                detected,
            )
            date_col_used = detected
        else:
            candidates = [c for c in df.columns if "date" in c.lower()]
            raise ValueError(
                f"Date column not found: {args.date_column}. "
                f"Candidates containing 'date': {candidates}"
            )
    dt_series = parse_date_series(df[date_col_used])


    lat_col = args.lat_column if args.lat_column in df.columns else None
    lon_col = args.lon_column if args.lon_column in df.columns else None
    if lat_col and lon_col:
        lat_vals = pd.to_numeric(df[lat_col], errors="coerce")
        lon_vals = pd.to_numeric(df[lon_col], errors="coerce")
    else:
        lat_vals = pd.Series([np.nan] * len(df))
        lon_vals = pd.Series([np.nan] * len(df))


    tz_col = args.timezone_column
    if tz_col is None:
        for col in df.columns:
            col_norm = normalize_name(col)
            if col_norm in ["timezone", "time_zone", "tz"]:
                tz_col = col
                break
    if tz_col and tz_col not in df.columns:
        logger.warning("Timezone column %s not found; ignoring.", tz_col)
        tz_col = None


    latlon_filled_from_cache = 0
    geocode_cache_path = Path(args.geocode_cache_path)
    geocode_map, geocode_default_country = load_zip_geocode_cache(geocode_cache_path)
    if (lat_col is None or lon_col is None) and not geocode_map:
        logger.warning(
            "Lat/Lon columns missing and geocode cache not found; UV features will be NaN."
        )

    job_id_col = resolve_weather_job_id_column(df, args.weather_job_id_column)
    zip_col = resolve_zip_column(df, args.zip_column)
    country_col = args.country_column if args.country_column in df.columns else None
    id_col = resolve_id_column(df, args.id_column)

    weather_job_ids = df[job_id_col] if job_id_col else pd.Series([None] * len(df))


    if job_id_col is None and id_col and Path(args.cats_to_job_path).exists():
        try:
            cats_map = pd.read_csv(args.cats_to_job_path)
            cats_id_col = resolve_id_column(cats_map, None)
            cats_date_col = auto_detect_date_column(cats_map.columns)
            cats_job_col = resolve_weather_job_id_column(cats_map, None)
            if cats_id_col and cats_date_col and cats_job_col:
                cats_map[cats_date_col] = parse_date_series(cats_map[cats_date_col]).dt.date
                lookup = {
                    (str(r[cats_id_col]), r[cats_date_col]): r[cats_job_col]
                    for _, r in cats_map.iterrows()
                }
                derived_job_ids = []
                for i in range(len(df)):
                    key = (str(df[id_col].iloc[i]), dt_series.iloc[i].date() if not pd.isna(dt_series.iloc[i]) else None)
                    derived_job_ids.append(lookup.get(key))
                weather_job_ids = pd.Series(derived_job_ids)
        except Exception as exc:
            error_counter[f"cats_to_job_map_error: {str(exc)}"] += 1

    job_to_zip: Dict[str, object] = {}
    if job_id_col or weather_job_ids.notna().any():
        manifest_path = Path(args.manifest_path)
        if manifest_path.exists():
            try:
                manifest = pd.read_csv(manifest_path, usecols=["weather_job_id", "zip"])
                job_to_zip = {
                    str(r["weather_job_id"]): r["zip"] for _, r in manifest.iterrows()
                }
            except Exception as exc:
                error_counter[f"manifest_read_error: {str(exc)}"] += 1

    def resolve_row_zip(i: int) -> Optional[str]:
        zip_val = None
        job_id_val = None
        if i < len(weather_job_ids):
            job_id_val = weather_job_ids.iloc[i]
        if job_id_val is not None and not pd.isna(job_id_val):
            zip_val = job_to_zip.get(str(job_id_val))
        if zip_val is None and zip_col:
            zip_val = df[zip_col].iloc[i]
        return normalize_zip(zip_val)

    if geocode_map:
        for i in range(len(df)):
            lat = lat_vals.iloc[i]
            lon = lon_vals.iloc[i]
            if valid_lat_lon(lat, lon):
                continue
            zip_norm = resolve_row_zip(i)
            if not zip_norm:
                continue
            country = None
            if country_col:
                country = normalize_country(df[country_col].iloc[i])
            if country is None:
                country = geocode_default_country
            key = (country, zip_norm)
            if key not in geocode_map:

                key = (None, zip_norm)
            if key in geocode_map:
                lat_vals.iloc[i], lon_vals.iloc[i] = geocode_map[key]
                latlon_filled_from_cache += 1


    tz_cache: Dict[str, timezone] = {}
    uv_dates: List[Optional[date]] = []
    missing_date_count = 0
    invalid_latlon_count = 0

    for idx in range(len(df)):
        dt = dt_series.iloc[idx]
        tz_str = None
        if tz_col:
            tz_val = df[tz_col].iloc[idx]
            if isinstance(tz_val, str) and tz_val.strip():
                tz_str = tz_val.strip()
        uv_date = compute_uv_date_for_row(dt, tz_str, tz_cache, error_counter)
        uv_dates.append(uv_date)

        lat = lat_vals.iloc[idx]
        lon = lon_vals.iloc[idx]
        if not valid_lat_lon(lat, lon):
            invalid_latlon_count += 1
        if uv_date is None:
            missing_date_count += 1


    moon_dates: List[Optional[date]] = [compute_moon_date_utc(dt) for dt in dt_series]


    uv_gap_fill_enabled = bool(args.uv_gap_fill_enabled) and not bool(args.uv_gap_fill_disabled)
    uv_gap_fill_max_days = int(args.uv_gap_fill_max_days)
    uv_source = args.uv_source
    uv_settings = None
    uv_variable_name = None

    if uv_source == "cams":

        cams_var_is_index = args.cams_variable_is_index
        if isinstance(cams_var_is_index, str):
            v = cams_var_is_index.strip().lower()
            if v in ["true", "1", "yes", "y"]:
                cams_var_is_index = True
            elif v in ["false", "0", "no", "n"]:
                cams_var_is_index = False
            elif v in ["none", "null", ""]:
                cams_var_is_index = None

        uv_settings = UVSettings(
            dataset=args.cams_dataset,
            variable=args.cams_variable,
            variable_is_index=cams_var_is_index,
            time_agg=args.cams_time_agg,
            time_list=CAMS_TIME_LIST,
            request_template=CAMS_REQUEST_TEMPLATE,
        )

        if uv_settings.variable_is_index is None:
            uv_settings.variable_is_index = infer_uv_index_flag(uv_settings.variable)

        uv_is_index = bool(uv_settings.variable_is_index)
        uv_variable_name = uv_settings.variable


        uv_fetch_available = True
        try:
            import cdsapi
        except Exception as exc:
            uv_fetch_available = False
            error_counter[f"cdsapi_import_error: {str(exc)}"] += 1
            logger.error("cdsapi not available; UV downloads will be skipped.")
    else:

        uv_is_index = True
        uv_variable_name = args.temis_prefix
        uv_fetch_available = True


    cache_dir = Path(args.cache_dir)
    uv_series_by_loc: Dict[Tuple[float, float], Dict[date, float]] = {}


    lat_round = lat_vals.round(args.round_decimals)
    lon_round = lon_vals.round(args.round_decimals)

    valid_mask = []
    for i in range(len(df)):
        if valid_lat_lon(lat_vals.iloc[i], lon_vals.iloc[i]) and uv_dates[i] is not None:
            valid_mask.append(True)
        else:
            valid_mask.append(False)

    valid_mask_series = pd.Series(valid_mask)
    df_valid = pd.DataFrame({
        "lat_round": lat_round,
        "lon_round": lon_round,
        "uv_date": uv_dates,
    })
    df_valid = df_valid[valid_mask_series]

    unique_locations = df_valid[["lat_round", "lon_round"]].drop_duplicates()

    uv_gap_filled_total = 0
    if uv_fetch_available:
        for _, loc in unique_locations.iterrows():
            rlat = float(loc["lat_round"])
            rlon = float(loc["lon_round"])
            loc_dates = df_valid[(df_valid["lat_round"] == rlat) & (df_valid["lon_round"] == rlon)][
                "uv_date"
            ]
            if loc_dates.empty:
                continue
            min_date = min(loc_dates)
            max_date = max(loc_dates)
            min_date = min_date - timedelta(days=14)
            if uv_source == "cams":
                uv_series = get_uv_series_cams(
                    rlat,
                    rlon,
                    min_date,
                    max_date,
                    uv_settings,
                    cache_dir,
                    error_counter,
                    logger,
                    args.uv_request_mode,
                )
            else:

                if uv_gap_fill_enabled:
                    max_date = max_date + timedelta(days=uv_gap_fill_max_days)
                uv_series, filled_count = get_uv_series_temis(
                    rlat,
                    rlon,
                    min_date,
                    max_date,
                    args.temis_base_url,
                    args.temis_prefix,
                    args.temis_coverage,
                    cache_dir,
                    error_counter,
                    logger,
                    uv_gap_fill_enabled,
                    uv_gap_fill_max_days,
                )
                uv_gap_filled_total += filled_count
            uv_series_by_loc[(rlat, rlon)] = uv_series


    uv_day0_vals = []
    uv_mean_lag1_2 = []
    uv_mean_lag1_3 = []
    uv_mean_lag1_7 = []
    uv_mean_lag1_13 = []
    uv_peak_lag1_7 = []
    uv_peak_lag1_14 = []

    for i in range(len(df)):
        if not valid_mask[i] or not uv_is_index:
            uv_day0_vals.append(np.nan)
            uv_mean_lag1_2.append(np.nan)
            uv_mean_lag1_3.append(np.nan)
            uv_mean_lag1_7.append(np.nan)
            uv_mean_lag1_13.append(np.nan)
            uv_peak_lag1_7.append(np.nan)
            uv_peak_lag1_14.append(np.nan)
            continue

        rlat = float(lat_round.iloc[i])
        rlon = float(lon_round.iloc[i])
        series = uv_series_by_loc.get((rlat, rlon), {})
        d0 = uv_dates[i]

        day0 = series.get(d0)
        lags_1_2 = [series.get(d0 - timedelta(days=k)) for k in range(1, 3)]
        lags_1_3 = [series.get(d0 - timedelta(days=k)) for k in range(1, 4)]
        lags_1_7 = [series.get(d0 - timedelta(days=k)) for k in range(1, 8)]
        lags_1_13 = [series.get(d0 - timedelta(days=k)) for k in range(1, 14)]
        lags_1_14 = [series.get(d0 - timedelta(days=k)) for k in range(1, 15)]

        uv_day0_vals.append(day0 if day0 is not None else np.nan)
        uv_mean_lag1_2.append(nanmean(lags_1_2))
        uv_mean_lag1_3.append(nanmean(lags_1_3))
        uv_mean_lag1_7.append(nanmean(lags_1_7))
        uv_mean_lag1_13.append(nanmean(lags_1_13))
        uv_peak_lag1_7.append(nanmax(lags_1_7))
        uv_peak_lag1_14.append(nanmax(lags_1_14))

    df[uv_mapping["uv_index_day0"]] = uv_day0_vals
    df[uv_mapping["uv_index_mean_lag1_2"]] = uv_mean_lag1_2
    df[uv_mapping["uv_index_mean_lag1_3"]] = uv_mean_lag1_3
    df[uv_mapping["uv_index_mean_lag1_7"]] = uv_mean_lag1_7
    df[uv_mapping["uv_index_mean_lag1_13"]] = uv_mean_lag1_13
    df[uv_mapping["uv_index_peak_lag1_7"]] = uv_peak_lag1_7
    df[uv_mapping["uv_index_peak_lag1_14"]] = uv_peak_lag1_14


    native_uv_cols = []
    if not uv_is_index:
        for col in uv_placeholders:
            if any(k in col.lower() for k in NATIVE_UV_KEYWORDS):
                native_uv_cols.append(col)
        if native_uv_cols:
            logger.warning("UV variable is not a UV index; filling native metric in: %s", native_uv_cols)
            native_vals = []
            for i in range(len(df)):
                if not valid_mask[i]:
                    native_vals.append(np.nan)
                    continue
                rlat = float(lat_round.iloc[i])
                rlon = float(lon_round.iloc[i])
                series = uv_series_by_loc.get((rlat, rlon), {})
                d0 = uv_dates[i]
                native_vals.append(series.get(d0, np.nan))
            for col in native_uv_cols:
                df[col] = native_vals
        else:
            error_counter["uv_variable_not_index_no_native_placeholder"] += 1


    moon_map: Dict[date, Tuple[float, float, str, int, int]] = {}
    for d in set([d for d in moon_dates if d is not None]):
        moon_map[d] = moon_phase(d)

    moon_fraction = []
    moon_angle = []
    moon_category = []
    is_full = []
    is_new = []

    for d in moon_dates:
        if d is None:
            moon_fraction.append(np.nan)
            moon_angle.append(np.nan)
            moon_category.append(np.nan)
            is_full.append(np.nan)
            is_new.append(np.nan)
        else:
            frac, ang, cat, full_flag, new_flag = moon_map[d]
            moon_fraction.append(frac)
            moon_angle.append(ang)
            moon_category.append(cat)
            is_full.append(full_flag)
            is_new.append(new_flag)

    df[moon_mapping["moon_phase_fraction_illuminated"]] = moon_fraction
    df[moon_mapping["moon_phase_angle_deg"]] = moon_angle
    df[moon_mapping["moon_phase_category"]] = moon_category
    df[moon_mapping["is_full_moon"]] = is_full
    df[moon_mapping["is_new_moon"]] = is_new


    uv_day0_col = uv_mapping["uv_index_day0"]
    lag_cols = [
        uv_mapping["uv_index_mean_lag1_2"],
        uv_mapping["uv_index_mean_lag1_3"],
        uv_mapping["uv_index_mean_lag1_7"],
        uv_mapping["uv_index_mean_lag1_13"],
        uv_mapping["uv_index_peak_lag1_7"],
        uv_mapping["uv_index_peak_lag1_14"],
    ]

    uv_coverage_day0 = float(df[uv_day0_col].notna().mean() * 100.0)
    uv_coverage_lags = float(df[lag_cols].notna().all(axis=1).mean() * 100.0)
    moon_coverage = float(df[moon_mapping["moon_phase_fraction_illuminated"]].notna().mean() * 100.0)

    report = {
        "number_of_rows": int(len(df)),
        "input_columns_count": input_columns_count,
        "date_column_requested": args.date_column,
        "date_column_used": date_col_used,
        "detected_uv_placeholder_columns": uv_placeholders,
        "detected_moon_placeholder_columns": moon_placeholders,
        "required_to_actual_column_mapping": required_to_actual,
        "number_unique_locations_used_for_uv": int(len(unique_locations)),
        "uv_coverage_percent_day0": uv_coverage_day0,
        "uv_coverage_percent_lag_features": uv_coverage_lags,
        "moon_phase_coverage_percent": moon_coverage,
        "rows_with_missing_or_invalid_latlon": int(invalid_latlon_count),
        "rows_latlon_filled_from_cache": int(latlon_filled_from_cache),
        "latlon_columns_present": bool(lat_col and lon_col),
        "zip_column_used": zip_col,
        "weather_job_id_column_used": job_id_col,
        "geocode_cache_path": str(geocode_cache_path),
        "geocode_default_country": geocode_default_country,
        "rows_with_missing_date": int(missing_date_count),
        "uv_date_basis": "timezone_column" if tz_col else "utc_or_naive_date",
        "uv_source": uv_source,
        "uv_variable": uv_variable_name,
        "uv_variable_is_index": uv_is_index,
        "uv_fetch_available": uv_fetch_available,
        "uv_mapping_notes": uv_mapping_notes,
        "uv_request_mode": args.uv_request_mode if uv_source == "cams" else "yearly",
        "temis_base_url": args.temis_base_url if uv_source == "temis_msr2" else None,
        "temis_prefix": args.temis_prefix if uv_source == "temis_msr2" else None,
        "temis_coverage": args.temis_coverage if uv_source == "temis_msr2" else None,
        "uv_gap_fill_enabled": uv_gap_fill_enabled if uv_source == "temis_msr2" else None,
        "uv_gap_fill_max_days": uv_gap_fill_max_days if uv_source == "temis_msr2" else None,
        "uv_gap_filled_count": uv_gap_filled_total if uv_source == "temis_msr2" else None,
        "top_error_messages": [
            {"message": msg, "count": cnt} for msg, cnt in error_counter.most_common(20)
        ],
    }

    if uv_source == "temis_msr2":
        report["uv_temis_note"] = (
            "TEMIS MSR-2 UV index is clear-sky at local solar noon; it does not account for clouds."
        )

    if not uv_is_index:
        report["uv_conversion_note"] = (
            "UV variable is not a UV index; uv_index_* columns set to NaN unless "
            "you implement a conversion. Native metric filled only if placeholders exist."
        )

    report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)


    output_path.parent.mkdir(parents=True, exist_ok=True)
    if input_is_xlsx:
        if output_path.suffix.lower() not in [".xlsx", ".xls"]:
            logger.warning("Output extension is not .xlsx/.xls; writing XLSX anyway.")
        df.to_excel(output_path, index=False, sheet_name=sheet_name)
    else:
        if output_path.suffix.lower() != ".csv":
            logger.warning("Output extension is not .csv; writing CSV anyway.")
        df.to_csv(output_path, index=False)

    logger.info("Enriched dataset saved to %s", output_path)
    logger.info("Enrichment report saved to %s", report_path)


if __name__ == "__main__":
    main()
